In [ ]:

# Instalar Gurobi (si Colab no lo tiene)
!pip -q install gurobipy

In [8]:
###############################################################
# CELDA 0
# CONFIGURACIÓN DEL EXPERIMENTO
###############################################################

import gurobipy as gp
from gurobipy import GRB
import time


rounds = 1
lanes = 5
bits = 4

TIME_LIMIT = 300


# False: Chi original
# True : Chi modificada
MODIFIED_CHI = False


print("="*70)
print("MILP KECCAK REDUCIDO - EXPERIMENTO CHI")
print("="*70)

print("Rondas:       ", rounds)
print("Lanes:        ", lanes,"x",lanes)
print("Bits/lane:    ", bits)
print("Estado total: ", lanes*lanes*bits,"bits")

print("="*70)

MILP KECCAK REDUCIDO - EXPERIMENTO CHI
Rondas:        1
Lanes:         5 x 5
Bits/lane:     4
Estado total:  100 bits


In [9]:
###############################################################
# CELDA 1
# ROTACIONES ρ + MODELO
###############################################################


rotation_offsets = [

    [0,36,3,41,18],
    [1,44,10,45,2],
    [62,6,43,15,61],
    [28,55,25,21,56],
    [27,20,39,8,14]

]


model = gp.Model("Keccak_Reducido")


model.Params.TimeLimit = TIME_LIMIT


print("Modelo creado")
print("Tabla de rotaciones cargada")

Set parameter TimeLimit to value 300
Modelo creado
Tabla de rotaciones cargada


In [10]:
###############################################################
# CELDA 2
# VARIABLES MILP
###############################################################

print()
print("="*70)
print("CREANDO VARIABLES")
print("="*70)


S = model.addVars(
    rounds+1,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="S"
)


C = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="C"
)


D = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="D"
)



ChiInput = model.addVars(
    rounds,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="ChiInput"
)



ChiActive = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="ChiActive"
)



ChiAND = model.addVars(
    rounds,
    lanes,
    lanes,
    bits,
    vtype=GRB.BINARY,
    name="ChiAND"
)



ThetaParity = model.addVars(
    rounds,
    lanes,
    bits,
    vtype=GRB.INTEGER,
    lb=0,
    ub=5,
    name="ThetaParity"
)


model.update()


print("Variables creadas")
print("Número de variables:", model.NumVars)


CREANDO VARIABLES
Variables creadas
Número de variables: 480


In [11]:
###############################################################
# CELDA 3
# FUNCIONES AUXILIARES
###############################################################


def xor2(a,b,out):

    model.addConstr(out <= a+b)

    model.addConstr(out >= a-b)

    model.addConstr(out >= b-a)

    model.addConstr(out <= 2-a-b)



def xor_list(values,out):

    temp = values[0]

    for i in range(1,len(values)):

        aux = model.addVar(
            vtype=GRB.BINARY
        )

        xor2(
            temp,
            values[i],
            aux
        )

        temp = aux


    model.addConstr(
        out == temp
    )



###############################################################
# Diferencia inicial y final
###############################################################

model.addConstr(

    gp.quicksum(
        S[0,x,y,z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    )
    >=1
)



model.addConstr(

    gp.quicksum(
        S[rounds,x,y,z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    )
    >=1
)



print("Restricciones iniciales creadas")

Restricciones iniciales creadas


In [12]:
###############################################################
# CELDA 4
# CONSTRUCCIÓN DE RONDA KECCAK
###############################################################


for r in range(rounds):


    print("Construyendo ronda:",r)


    ###########################################################
    # THETA
    ###########################################################

    for x in range(lanes):

        for z in range(bits):


            columna = gp.quicksum(

                S[r,x,y,z]

                for y in range(lanes)

            )


            model.addConstr(

                columna

                ==
                
                C[r,x,z]
                +
                2*ThetaParity[r,x,z]

            )



    for x in range(lanes):

        for z in range(bits):

            xor2(

                C[r,(x-1)%lanes,z],

                C[r,(x+1)%lanes,(z-1)%bits],

                D[r,x,z]

            )



    ###########################################################
    # RHO + PI
    ###########################################################

    for x in range(lanes):

        for y in range(lanes):


            xp=(x+3*y)%lanes

            yp=y


            shift=rotation_offsets[x][y]%bits


            for z in range(bits):


                zp=(z-shift)%bits


                temp=model.addVar(
                    vtype=GRB.BINARY
                )


                xor2(

                    S[r,x,y,zp],

                    D[r,x,zp],

                    temp

                )


                model.addConstr(

                    ChiInput[r,xp,yp,z]

                    ==
                    
                    temp

                )



    ###########################################################
    # ACTIVIDAD CHI
    ###########################################################

    for y in range(lanes):

        for z in range(bits):


            for x in range(lanes):

                model.addConstr(

                    ChiActive[r,y,z]
                    >=
                    ChiInput[r,x,y,z]

                )


            model.addConstr(

                ChiActive[r,y,z]

                <=

                gp.quicksum(
                    ChiInput[r,x,y,z]
                    for x in range(lanes)
                )

            )



    ###########################################################
    # CHI
    ###########################################################

    for y in range(lanes):

        for z in range(bits):

            for x in range(lanes):


                x1=(x+1)%lanes
                x2=(x+2)%lanes


                model.addConstr(

                    ChiAND[r,x,y,z]

                    <=

                    ChiInput[r,x2,y,z]

                )


                model.addConstr(

                    ChiAND[r,x,y,z]

                    <=

                    1-ChiInput[r,x1,y,z]

                )


                model.addConstr(

                    ChiAND[r,x,y,z]

                    >=

                    ChiInput[r,x2,y,z]
                    -
                    ChiInput[r,x1,y,z]

                )



                temp=model.addVar(
                    vtype=GRB.BINARY
                )


                xor2(

                    ChiInput[r,x,y,z],

                    ChiAND[r,x,y,z],

                    temp

                )


                if MODIFIED_CHI:

                    xor2(

                        temp,

                        ChiInput[r,(x+3)%lanes,y,z],

                        S[r+1,x,y,z]

                    )

                else:

                    model.addConstr(

                        S[r+1,x,y,z]
                        ==
                        temp

                    )



model.update()


print("="*70)
print("RONDA COMPLETADA")
print("="*70)

print("Variables:",model.NumVars)
print("Restricciones:",model.NumConstrs)

Construyendo ronda: 0
RONDA COMPLETADA
Variables: 680
Restricciones: 1522


In [13]:
###############################################################
# CELDA 5
# OBJETIVO + OPTIMIZACIÓN
###############################################################


objetivo = gp.quicksum(

    ChiActive[r,y,z]

    for r in range(rounds)

    for y in range(lanes)

    for z in range(bits)

)


model.setObjective(
    objetivo,
    GRB.MINIMIZE
)


model.update()


print("="*70)
print("RESOLVIENDO MODELO MILP")
print("="*70)

print("Variables:",model.NumVars)
print("Restricciones:",model.NumConstrs)



inicio=time.time()

model.optimize()

tiempo=time.time()-inicio



print("="*70)
print("RESULTADOS")
print("="*70)



if model.SolCount > 0:

    print("Estado OPTIMAL")

    print(
        "Mínimo Chi activo:",
        model.ObjVal
    )

    print(
        "Tiempo:",
        tiempo
    )


    for r in range(rounds):

        cuenta=0

        for y in range(lanes):

            for z in range(bits):

                if ChiActive[r,y,z].X >0.5:

                    cuenta+=1


        print(
            "Ronda",
            r,
            ":",
            cuenta
        )

RESOLVIENDO MODELO MILP
Variables: 680
Restricciones: 1522
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 1522 rows, 680 columns and 4400 nonzeros (Min)
Model fingerprint: 0xd87ee3db
Model has 20 linear objective coefficients
Variable types: 0 continuous, 680 integer (660 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+00, 2e+00]

Presolve removed 620 rows and 300 columns
Presolve time: 0.02s
Presolved: 902 rows, 380 columns, 2980 nonzeros
Variable types: 0 continuous, 380 integer (360 binary)
Found heuristic solution: objective 20.0000000
Found heuristic solution: objective 19.0000000
Found heuristic solution: objective 